In [1]:
import ee
import geemap
import math 
import os

In [2]:


# 设置代理（替换成你的代理地址）
os.environ["HTTP_PROXY"] = "http://127.0.0.1:7897"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:7897"

In [3]:

ee.Authenticate()
ee.Initialize()



Successfully saved authorization token.


In [4]:
Map = geemap.Map()

In [5]:
ee.Initialize()

In [6]:
from utils.utils import *
from utils.ee_utils import *
from utils.utils import TextColors as c

In [7]:
# df = read_csv("LUCAS_2015_all.csv")
df = pd.read_csv('CN-SOC-3500_new.csv')
df = df.iloc[1700:]
df 


,Point_id,Latitude,Longitude,Year,SOC,CLCD,BD_g_cm3,Sand_%,Silt_%,Clay_%
1700,1702,34.546633,116.782467,2014,3.71,Cropland,1.546667,27.686667,56.633333,15.620000
1701,1703,34.429058,119.887362,2014,5.63,Other,1.470000,12.396667,66.293333,21.223333
1702,1704,34.725438,116.482877,2014,7.47,Forest,1.476000,21.724000,65.054000,13.232000
1703,1705,34.819393,116.449951,2014,7.34,Forest,1.505000,23.353333,65.113333,11.531667
1704,1706,32.756240,120.676510,2014,8.87,Forest,1.482958,16.600000,69.857500,13.602500
...,...,...,...,...,...,...,...,...,...,...
3450,3467,30.217806,105.875833,2014,1.60,Grassland,1.500000,53.500000,31.190000,15.580000
3451,3468,29.760528,106.431111,2014,9.54,Forest,1.340000,60.175000,21.670000,18.060000
3452,3469,29.145667,107.194500,2014,19.17,Forest,1.367500,36.260000,39.835000,23.900000
3453,3470,30.012472,108.483694,2014,33.00,Cropland,1.210000,22.805000,49.585000,27.535000


In [8]:
DATASET = 'CN' #'LUCAS'
if DATASET == 'CN':
 create_folder_if_not_exists('l8_images_newCN')
 DFOLDER = 'l8_images_newCN//'
if DATASET == 'RaCA':
 create_folder_if_not_exists('l8_images_us')
 DFOLDER = 'l8_images_us//'

if DATASET == 'LUCAS':
 create_folder_if_not_exists('l8_images')
 DFOLDER = 'l8_images//'

Folder "l8_images_newCN" created in the current working directory.


In [9]:
"""
多年裸土合成版 Landsat-8 影像下载脚本（放宽筛选条件版）

主要特性：
1. 2013–2017 多年 Landsat-8 C2 L2
2. 云/阴影/雪 mask
3. 像元级 NDVI 裸土筛选（默认 NDVI < 0.35，可视情况再调）
4. 多景 median 合成得到 bare-soil composite
5. 输出：原始 SR_B2-7 + 指数 + Tasseled Cap + 简单矿物指数 + 纹理 + DEM/slope

使用前确保：
- 已执行 ee.Initialize()
- 已加载 df（包含 Point_id, Latitude, Longitude 列）
- 已设置 DFOLDER 输出路径（末尾加 / 或 // 均可）
"""

import ee
import geemap
import pandas as pd

# ============== 1. 配置区域 & 参数 ==============

# Landsat-8 C2 L2 影像集
IMAGE_COLLECTION = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')

# 多年份范围（建议和 SOC 样本年份相匹配）
GLOBAL_START = '2013-01-01'
GLOBAL_END   = '2017-12-31'

# 冬季 / 非冬季（可以根据区域调整）
WINTER_MONTHS     = [12, 1, 2]
NON_WINTER_MONTHS = [3,4,5,6,7,8,9,10,11]

# 输出 band 列表（你可再扩展/删减）
BANDS = [
    'SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7',  # 关键光学波段
    'NDVI','NDMI','BSI','NBR2',                      # 常用指数
    'TCB','TCG','TCW',                               # Tasseled Cap
    'clayIndex','ferrousIndex','carbonateIndex','rockOutcropIndex',
    'tex_contrast','tex_entropy',                    # 纹理
    'DEM','slope'                                    # topo（如你已放静态分支，可移除）
]

# DFOLDER 需要你在外面提前定义，例如：
# DFOLDER = 'l8_images_CN/'
# 并且确保这个文件夹已经存在（可以用 os.makedirs）


# ============== 2. 工具函数 ==============

def get_square_roi(lat, lon, roi_size=1920, return_gee_object=True):
    """
    根据中心点生成正方形 ROI，roi_size 单位：米
    """
    point = ee.Geometry.Point([lon, lat])
    half = roi_size / 2.0
    roi = point.buffer(half).bounds()  # 近似正方形
    return roi if return_gee_object else None


def radiometric_correction(img):
    """
    Landsat-8 C2 L2 反射率 & 地表温度缩放。
    完全 server-side 写法。
    """
    # SR_B2-7 反射率缩放
    optical = img.select(['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7']) \
                 .multiply(0.0000275).add(-0.2)
    img = img.addBands(optical, overwrite=True)

    # ST_B10 -> LST（Kelvin），使用服务器端判断
    has_st = img.bandNames().contains('ST_B10')
    lst    = img.select('ST_B10').multiply(0.00341802).add(149.0).rename('LST')
    img    = ee.Image(ee.Algorithms.If(has_st, img.addBands(lst, overwrite=True), img))

    return img


def get_cloud_mask(img):
    """
    从 QA_PIXEL 生成云/阴影/雪掩膜。
    返回 [cloud, shadow, combined, snow]（1=有云/阴影/雪）。
    """
    qa = img.select('QA_PIXEL')
    cloud_bit  = 1 << 3
    shadow_bit = 1 << 4
    snow_bit   = 1 << 5

    cloud  = qa.bitwiseAnd(cloud_bit).gt(0)
    shadow = qa.bitwiseAnd(shadow_bit).gt(0)
    snow   = qa.bitwiseAnd(snow_bit).gt(0)

    combined = cloud.Or(shadow)  # 云+阴影组合
    return [cloud.rename('cloud'),
            shadow.rename('shadow'),
            combined.rename('cloud_shadow'),
            snow.rename('snow')]


def get_snow_mask(img):
    """
    简单雪掩膜：直接用 QA_PIXEL bit 5
    """
    qa = img.select('QA_PIXEL')
    snow_bit = 1 << 5
    snow = qa.bitwiseAnd(snow_bit).gt(0).rename('snow')
    return snow


def get_not_nulls_ratio(img, roi):
    """
    计算 img 在 ROI 中非空像元比例（完全 server-side，不用 getInfo）
    用第一条 band 的 mask 代表整体。
    返回：ee.Number
    """
    band = img.select(0)
    mask = band.mask()

    # 非空像元数
    non_null = mask.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=30,
        maxPixels=1e9
    ).values().get(0)  # ee.Number

    # 总像元数（用常数影像 1 再 sum）
    ones = ee.Image.constant(1).clip(roi)
    total = ones.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=30,
        maxPixels=1e9
    ).values().get(0)

    ratio = ee.Number(non_null).divide(ee.Number(total).max(1))
    return ratio


def add_topo():
    """
    加 DEM + slope。
    若你已经有 topo 栅格，可以改用你自己的。
    """
    srtm  = ee.Image('USGS/SRTMGL1_003').rename('DEM')
    slope = ee.Terrain.slope(srtm).rename('slope')
    return srtm.addBands(slope)


def add_mineral_indices(img):
    """
    在已有 SR_B* 的基础上添加：
    - NDVI, NDMI, BSI, NBR2
    - Tasseled Cap (TCB, TCG, TCW)
    - 简单矿物指数 clay/ferrous/carbonate/rockOutcrop（示例，可替换为你原来的）
    - 纹理 tex_contrast, tex_entropy（基于 B6）
    """
    b2 = img.select('SR_B2')
    b3 = img.select('SR_B3')
    b4 = img.select('SR_B4')
    b5 = img.select('SR_B5')
    b6 = img.select('SR_B6')
    b7 = img.select('SR_B7')

    # === 基础指数 ===
    ndvi = img.normalizedDifference(['SR_B5','SR_B4']).rename('NDVI')
    ndmi = img.normalizedDifference(['SR_B5','SR_B6']).rename('NDMI')
    nbr2 = img.normalizedDifference(['SR_B6','SR_B7']).rename('NBR2')
    bsi  = (b6.add(b4).subtract(b5.add(b2))) \
           .divide(b6.add(b4).add(b5).add(b2)) \
           .rename('BSI')

    # === Tasseled Cap（Landsat-8 一套常用系数）===
    # [B2,B3,B4,B5,B6,B7]
    tc_matrix = ee.Array([
      [0.3029,  0.2786,  0.4733,  0.5599,  0.5080,  0.1872],   # Brightness
      [-0.2941, -0.2430, -0.5424, 0.7276,  0.0713, -0.1608],   # Greenness
      [0.1511,  0.1973,  0.3283,  0.3407, -0.7117, -0.4559]    # Wetness
    ])

    array_image_1d = img.select(['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7']).toArray()
    array_image_2d = array_image_1d.toArray(1)

    tc_image = ee.Image(tc_matrix) \
        .matrixMultiply(array_image_2d) \
        .arrayProject([0]) \
        .arrayFlatten([['TCB','TCG','TCW']])

    # === 简单矿物指数示例（你可换回你已有 clayIndex 等实现）===
    clayIndex        = b6.divide(b7).rename('clayIndex')
    ferrousIndex     = b4.divide(b2).rename('ferrousIndex')
    carbonateIndex   = b3.divide(b5).rename('carbonateIndex')
    rockOutcropIndex = b7.divide(b5).rename('rockOutcropIndex')

    # === 纹理：基于 B6 ===
    scaled = b6.multiply(255).toInt()
    glcm = scaled.glcmTexture(size=3)
    tex_contrast = glcm.select('SR_B6_contrast').rename('tex_contrast')
    tex_entropy  = glcm.select('SR_B6_ent').rename('tex_entropy')  # 修复：GEE 返回 ent 而不是 entropy

    return img.addBands([
        ndvi, ndmi, nbr2, bsi,
        tc_image.select(['TCB','TCG','TCW']),
        clayIndex, ferrousIndex, carbonateIndex, rockOutcropIndex,
        tex_contrast, tex_entropy
    ])


# ============== 3. 多年裸土预处理主流程 ==============

def preprocess_l8(img, roi):
    """
    对单景 Landsat8 做：
    - clip 到 ROI
    - radiometric_correction
    - 云/阴影/雪 mask
    - 统计 无云无雪 后的有效像元比例
    - 再做裸土 mask（NDVI < 0.35，可视需要调为 0.4）
    """
    img = img.clip(roi)
    img = radiometric_correction(img)

    # 1. 云/阴影/雪 mask
    cloud, shadow, cloud_shadow, snow = get_cloud_mask(img)
    snow = get_snow_mask(img)   # 如果你想用另外的雪判据，可以替换上面一行
    mask = cloud_shadow.eq(0).And(snow.eq(0))
    img = img.updateMask(mask)

    # 2. 在“无云无雪”基础上统计 not_null_pixels
    img = img.set('not_null_pixels', get_not_nulls_ratio(img, roi))

    # 3. 再算 NDVI / NDWI
    ndvi = img.normalizedDifference(['SR_B5','SR_B4']).rename('NDVI_tmp')
    ndwi = img.normalizedDifference(['SR_B3','SR_B5']).rename('NDWI_tmp')
    img = img.addBands([ndvi, ndwi])

    # 4. 裸土像元 mask（先放宽：只用 NDVI，阈值 0.35）
    bare_mask = ndvi.lt(0.40)
    # 若你想兼顾水体，可进一步：bare_mask = ndvi.lt(0.35).And(ndwi.lt(0.1))
    img = img.updateMask(bare_mask)

    return img


def build_l8_collection_for_roi(roi):
    """
    构建多年份 Landsat8 裸土候选集合
    过滤条件：
    - 时间/区域过滤
    - preprocess_l8
    - not_null_pixels > 0.1（即无云无雪有效像元比例 >10%）
    """
    col = (IMAGE_COLLECTION
           .filterDate(GLOBAL_START, GLOBAL_END)
           .filterBounds(roi))

    col = col.map(lambda i: preprocess_l8(i, roi))

    # 原来是 0.3，这里放宽到 0.1
    col = col.filter(ee.Filter.gt('not_null_pixels', 0.05))
    return col


def split_winter_nonwinter(col):
    """按月份划分冬季 / 非冬季"""
    def add_month(img):
        return img.set('month', ee.Date(img.get('system:time_start')).get('month'))

    col = col.map(add_month)
    winter_col    = col.filter(ee.Filter.inList('month', WINTER_MONTHS))
    nonwinter_col = col.filter(ee.Filter.inList('month', NON_WINTER_MONTHS))
    return winter_col, nonwinter_col


def build_bare_soil_composite(roi):
    """
    针对一个 ROI：
    - 优先用冬季裸土影像：median 合成
    - 若冬季无数据，则用非冬季裸土影像：median 合成
    返回：合成影像, 使用季节标签
    """
    col = build_l8_collection_for_roi(roi)
    winter_col, nonwinter_col = split_winter_nonwinter(col)

    # 一次性获取两个集合的大小，减少 API 调用
    sizes = ee.Dictionary({
        'winter': winter_col.size(),
        'nonwinter': nonwinter_col.size()
    }).getInfo()

    winter_size    = sizes['winter']
    nonwinter_size = sizes['nonwinter']

    if winter_size > 0:
        bare = winter_col.median().clip(roi)
        season_tag = 'winter'
    elif nonwinter_size > 0:
        bare = nonwinter_col.median().clip(roi)
        season_tag = 'non_winter'
    else:
        bare = None
        season_tag = 'none'

    return bare, season_tag


# ============== 4. 主循环：逐点导出 patch ==============

print(f"开始处理 {len(df)} 个采样点...")
success_count = 0
skip_count = 0
error_count = 0

for idx, row in df.iterrows():
    try:
        point_id = row['Point_id']
        lat = row['Latitude']
        lon = row['Longitude']

        print(f"[{idx+1}/{len(df)}] Point {point_id} building bare-soil composite ... ", end='')

        roi = get_square_roi(lat, lon, roi_size=1920, return_gee_object=True)

        l8_bare, season_used = build_bare_soil_composite(roi)

        if l8_bare is None:
            print("NO VALID IMAGE (skip)")
            skip_count += 1
            continue

        print(f"OK ({season_used}), adding topo & indices ... ", end='')

        topo = add_topo().clip(roi)
        img_with_idx = add_mineral_indices(l8_bare).addBands(topo)

        # 多景 median 之后，system:time_start 通常不可用，这里用固定命名
        name = f"{point_id}_{season_used}.tif"
        print(f"exporting {name} ...")

        geemap.download_ee_image(
            img_with_idx.select(BANDS),
            DFOLDER + name,
            crs='EPSG:3857',
            scale=30,
            region=roi
        )

        success_count += 1
        print(f"✓ 成功 (累计: {success_count}成功, {skip_count}跳过, {error_count}错误)")

    except Exception as e:
        error_count += 1
        print(f"✗ 错误: {str(e)}")
        continue

    # 调试时可限制前 N 个点：
    # if idx == 20:
    #     break

print(f"\n完成！总计: {success_count}成功, {skip_count}跳过, {error_count}错误")


开始处理 1755 个采样点...
[1701/1755] Point 1702 building bare-soil composite ... OK (winter), adding topo & indices ... exporting 1702_winter.tif ...


1702_winter.tif: |          | 0.00/1.02M (raw) [  0.0%] in 00:00 (eta:     ?)

There is no STAC entry for: None


✓ 成功 (累计: 1成功, 0跳过, 0错误)
[1702/1755] Point 1703 building bare-soil composite ... OK (winter), adding topo & indices ... exporting 1703_winter.tif ...


1703_winter.tif: |          | 0.00/1.04M (raw) [  0.0%] in 00:00 (eta:     ?)

...

✓ 成功 (累计: 1738成功, 0跳过, 15错误)
[3454/1755] Point 3470 building bare-soil composite ... OK (winter), adding topo & indices ... exporting 3470_winter.tif ...


3470_winter.tif: |          | 0.00/932k (raw) [  0.0%] in 00:00 (eta:     ?)

✓ 成功 (累计: 1739成功, 0跳过, 15错误)
[3455/1755] Point 3471 building bare-soil composite ... OK (winter), adding topo & indices ... exporting 3471_winter.tif ...


3471_winter.tif: |          | 0.00/970k (raw) [  0.0%] in 00:00 (eta:     ?)

✓ 成功 (累计: 1740成功, 0跳过, 15错误)

完成！总计: 1740成功, 0跳过, 15错误
